# Text Cleaning & Preprocessing Pipeline

## Week 1: Data Preprocessing Validation

### Objective
This notebook demonstrates the text preprocessing pipeline used for cleaning and preparing
citizen grievance data for NLP modeling.

We will:
- Apply cleaning and preprocessing functions
- Compare raw vs cleaned vs processed text
- Validate transformations
- Ensure data is ready for feature extraction and modeling

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.preprocessing.cleaning import TextCleaner

In [2]:
df = pd.read_csv("../data/raw/civic_complaints.csv")

df.head()

,text,department,sentiment
0,"No water supply in our area for last 3 days, p...",Water,Urgent
1,Water is coming very dirty and smells bad,Water,Negative
2,Low water pressure in the morning hours,Water,Negative
3,Thank you for restoring water supply quickly,Water,Positive
4,Pipeline leakage near main road causing wastage,Water,Urgent


In [3]:
cleaner = TextCleaner()

In [4]:
sample_text = df["text"].iloc[0]

print("Original Text:\n", sample_text)
print("\nLowercase:\n", cleaner.to_lower(sample_text))
print("\nRemove URLs:\n", cleaner.remove_urls(sample_text))
print("\nRemove Special Characters:\n", cleaner.remove_special_characters(sample_text))

Original Text:
 No water supply in our area for last 3 days, please resolve urgently

Lowercase:
 no water supply in our area for last 3 days, please resolve urgently

Remove URLs:
 No water supply in our area for last 3 days, please resolve urgently

Remove Special Characters:
 No water supply in our area for last 3 days please resolve urgently


In [5]:
df["clean_text"] = df["text"].apply(cleaner.clean_text)

df[["text", "clean_text"]].head()

,text,clean_text
0,"No water supply in our area for last 3 days, p...",no water supply in our area for last days plea...
1,Water is coming very dirty and smells bad,water is coming very dirty and smells bad
2,Low water pressure in the morning hours,low water pressure in the morning hours
3,Thank you for restoring water supply quickly,thank you for restoring water supply quickly
4,Pipeline leakage near main road causing wastage,pipeline leakage near main road causing wastage


In [6]:
df["processed_text"] = df["clean_text"].apply(
    lambda x: cleaner.preprocess_text(x, use_spacy=True)
)

df[["clean_text", "processed_text"]].head()

,clean_text,processed_text
0,no water supply in our area for last days plea...,water supply area last day resolve urgently
1,water is coming very dirty and smells bad,water come dirty smell bad
2,low water pressure in the morning hours,low water pressure morning hour
3,thank you for restoring water supply quickly,restore water supply quickly
4,pipeline leakage near main road causing wastage,pipeline leakage near main road cause wastage


In [7]:
for i in range(5):
    print("Original :", df["text"].iloc[i])
    print("Cleaned  :", df["clean_text"].iloc[i])
    print("Processed:", df["processed_text"].iloc[i])
    print("-" * 80)

Original : No water supply in our area for last 3 days, please resolve urgently
Cleaned  : no water supply in our area for last days please resolve urgently
Processed: water supply area last day resolve urgently
--------------------------------------------------------------------------------
Original : Water is coming very dirty and smells bad
Cleaned  : water is coming very dirty and smells bad
Processed: water come dirty smell bad
--------------------------------------------------------------------------------
Original : Low water pressure in the morning hours
Cleaned  : low water pressure in the morning hours
Processed: low water pressure morning hour
--------------------------------------------------------------------------------
Original : Thank you for restoring water supply quickly
Cleaned  : thank you for restoring water supply quickly
Processed: restore water supply quickly
--------------------------------------------------------------------------------
Original : Pipeline lea

In [8]:
df["original_tokens"] = df["text"].apply(lambda x: len(str(x).split()))
df["processed_tokens"] = df["processed_text"].apply(lambda x: len(str(x).split()))

df[["original_tokens", "processed_tokens"]].head()

,original_tokens,processed_tokens
0,13,7
1,8,5
2,7,5
3,7,4
4,7,7


In [9]:
print("Sample Processed Text:\n")
print(df["processed_text"].iloc[0])

Sample Processed Text:

water supply area last day resolve urgently


In [10]:
hinglish_examples = [
    "paani nahi aa raha",
    "bijli chali gayi",
    "kachra bahut hai"
]

for text in hinglish_examples:
    print("Original :", text)
    print("Processed:", cleaner.preprocess_text(text))
    print("-" * 50)

Original : paani nahi aa raha
Processed: water nahi aa raha
--------------------------------------------------
Original : bijli chali gayi
Processed: electricity chali gayi
--------------------------------------------------
Original : kachra bahut hai
Processed: garbage bahut hai
--------------------------------------------------


In [11]:
edge_cases = [
    "!!!",
    "1234567890",
    "Visit http://abc.com",
    "   ",
    "PLS HELP!!!"
]

for text in edge_cases:
    print("Original :", repr(text))
    print("Processed:", repr(cleaner.preprocess_text(text)))
    print("-" * 50)

Original : '!!!'
Processed: ''
--------------------------------------------------
Original : '1234567890'
Processed: ''
--------------------------------------------------
Original : 'Visit http://abc.com'
Processed: 'visit'
--------------------------------------------------
Original : '   '
Processed: ''
--------------------------------------------------
Original : 'PLS HELP!!!'
Processed: 'help'
--------------------------------------------------


In [12]:
df.to_csv("../data/interim/cleaned_data_from_notebook.csv", index=False)

## Observations & Validation

1. **Noise Removal**
   - URLs, special characters, and numbers are successfully removed.

2. **Normalization**
   - Text is consistently lowercased.
   - Hinglish terms are partially normalized.

3. **Stopword Removal**
   - Common filler words like "sir", "please", "kindly" are removed.

4. **Lemmatization**
   - Words are reduced to base forms, improving model generalization.

5. **Token Reduction**
   - Processed text is significantly shorter and more meaningful.

6. **Edge Case Handling**
   - Empty and noisy inputs are handled gracefully.

---

## Conclusion

The preprocessing pipeline is robust and suitable for:
- Feature extraction (TF-IDF, embeddings)
- Machine learning models
- Real-world noisy civic complaint data